Section	Nội dung
1. Load Data	Đọc Gold Weekly
2. Temporal Split	Xác định mốc Train/Test
3. SKU Selection	Chọn Top 20 chỉ từ Train
4. Data Preparation	Chuẩn bị X, y và kiểm tra dữ liệu
5. Baseline	Dự báo bằng Lag_1W
6. Model Training	Random Forest và XGBoost
7. Evaluation	MAE, RMSE, WAPE, Bias
8. Comparison	So sánh thực nghiệm cũ và mới
9. Error Analysis	Phân tích SKU dự báo kém

Notebook: Model Training V2
Section 1. Load Data
Mục tiêu: Nạp Gold Weekly và kiểm tra cấu trúc dữ liệu đầu vào.

Markdown cell
# 04 - Model Training V2

## 1. Load Data

Load weekly demand features from the Gold layer.
This dataset contains historical demand, engineered features, and
the next-week demand target for each SKU.

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

BASE_DIR = Path("..")
GOLD_DIR = BASE_DIR / "data" / "3_gold"

df_gold = pl.read_parquet(GOLD_DIR / "weekly_demand_features.parquet")

print(f"Dataset shape: {df_gold.shape}")
print(f"Date range: {df_gold['Week_Start'].min()} -> {df_gold['Week_Start'].max()}")
print(f"Unique SKUs: {df_gold['StockCode'].n_unique()}")
print(f"Unique weeks: {df_gold['Week_Start'].n_unique()}")

display(df_gold.head())

Dataset shape: (486684, 19)
Date range: 2010-01-04 00:00:00 -> 2011-11-28 00:00:00
Unique SKUs: 4916
Unique weeks: 99


Week_Start,StockCode,Weekly_Demand,Weekly_Revenue,Avg_UnitPrice,Description,Target_Demand_Weekly,Lag_1W,Lag_2W,Lag_4W,Rolling_Mean_4W,Rolling_Std_4W,Month,Quarter,Year,Is_Peak_Season,Demand_Trend_Ratio,Demand_CV,Price_Ratio_Vs_Avg
datetime[μs],str,i64,f64,f64,str,i64,i64,i64,i64,f64,f64,i8,i8,i32,i32,f64,f64,f64
2010-01-04 00:00:00,"""10002""",15,12.75,0.85,"""INFLATABLE POLITICAL GLOBE""",191,14,37,141,54.0,58.759396,1,1,2010,0,0.254545,1.068353,0.813552
2010-01-11 00:00:00,"""10002""",191,162.35,0.85,"""INFLATABLE POLITICAL GLOBE""",41,15,14,24,22.5,10.661457,1,1,2010,0,0.638298,0.453679,0.813552
2010-01-18 00:00:00,"""10002""",41,35.66,1.012,"""INFLATABLE POLITICAL GLOBE""",44,191,15,37,64.25,85.164057,1,1,2010,0,2.927203,1.305196,1.020494
2010-01-25 00:00:00,"""10002""",44,38.21,0.985,"""INFLATABLE POLITICAL GLOBE""",16,41,191,14,65.25,84.759955,1,1,2010,0,0.618868,1.279396,1.024608
2010-02-01 00:00:00,"""10002""",16,14.41,1.12,"""INFLATABLE POLITICAL GLOBE""",43,44,41,15,72.75,79.901502,2,1,2010,0,0.59661,1.08341,1.21178


## 2. Temporal Train/Test Split

Split the dataset chronologically using unique weekly timestamps.

- Training set: earliest 80% of weeks.
- Testing set: latest 20% of weeks.

The test period must always occur after the training period.

In [2]:
weeks = (
    df_gold
    .select("Week_Start")
    .unique()
    .sort("Week_Start")
    .to_series()
    .to_list()
)

split_idx = int(len(weeks) * 0.8)
split_week = weeks[split_idx]

print(f"Total weeks: {len(weeks)}")
print(f"Split week: {split_week}")

print(f"Train period: {weeks[0]} -> {weeks[split_idx - 1]}")
print(f"Test period:  {weeks[split_idx]} -> {weeks[-1]}")

Total weeks: 99
Split week: 2011-07-18 00:00:00
Train period: 2010-01-04 00:00:00 -> 2011-07-11 00:00:00
Test period:  2011-07-18 00:00:00 -> 2011-11-28 00:00:00


## 3. Train-based SKU Selection

Select the top 20 SKUs based only on their total historical demand
during the training period.

SKUs with zero training demand are excluded from this forecasting
experiment because the objective is to forecast established products
with observed demand history.

In [3]:
train_history = df_gold.filter(
    pl.col("Week_Start") < split_week
)

sku_ranking = (
    train_history
    .group_by("StockCode")
    .agg([
        pl.col("Weekly_Demand").sum().alias("Train_Total_Demand"),
        pl.col("Week_Start").n_unique().alias("Active_Weeks")
    ])
    .filter(pl.col("Train_Total_Demand") > 0)
    .sort("Train_Total_Demand", descending=True)
)

top_skus = sku_ranking.head(20).select("StockCode")

print("Selected SKUs:", top_skus.height)
display(sku_ranking.head(20))

print(
    "SKU 23843 selected:",
    "23843" in top_skus["StockCode"].to_list()
)

Selected SKUs: 20


StockCode,Train_Total_Demand,Active_Weeks
str,i64,u32
"""84077""",86004,79
"""21212""",78533,79
"""85123A""",76236,79
"""23166""",75589,79
"""85099B""",70976,79
…,…,…
"""21232""",30690,79
"""21982""",30421,79
"""22616""",29853,79


SKU 23843 selected: False


## 4. Data Preparation

Filter the dataset using the selected training-based SKU list.

Separate features and target:
- Features: historical demand, rolling statistics, price and calendar features.
- Target: next week's demand.

Use the same features and observations for all models to ensure a fair comparison.

In [4]:
features = [
    "Lag_1W", "Lag_2W", "Lag_4W",
    "Rolling_Mean_4W", "Rolling_Std_4W",
    "Demand_Trend_Ratio", "Demand_CV",
    "Price_Ratio_Vs_Avg", "Avg_UnitPrice",
    "Month", "Quarter", "Year", "Is_Peak_Season"
]

target = "Target_Demand_Weekly"

df_sub = df_gold.join(top_skus, on="StockCode", how="inner")

train_df = df_sub.filter(pl.col("Week_Start") < split_week)
test_df = df_sub.filter(pl.col("Week_Start") >= split_week)

# Kiểm tra null trước khi huấn luyện
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(
    train_df.select([
        pl.col(features + [target]).null_count()
    ])
)

Train shape: (1580, 19)
Test shape: (400, 19)


Lag_1W,Lag_2W,Lag_4W,Rolling_Mean_4W,Rolling_Std_4W,Demand_Trend_Ratio,Demand_CV,Price_Ratio_Vs_Avg,Avg_UnitPrice,Month,Quarter,Year,Is_Peak_Season,Target_Demand_Weekly
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
train_pd = train_df.select(features + [target]).to_pandas()
test_pd = test_df.select(features + [target]).to_pandas()

train_pd = train_pd.replace([np.inf, -np.inf], np.nan)
test_pd = test_pd.replace([np.inf, -np.inf], np.nan)

print("Train missing values:")
display(train_pd.isna().sum())

print("Test missing values:")
display(test_pd.isna().sum())

Train missing values:


Lag_1W                  0
Lag_2W                  0
Lag_4W                  0
Rolling_Mean_4W         0
Rolling_Std_4W          0
Demand_Trend_Ratio      0
Demand_CV               0
Price_Ratio_Vs_Avg      0
Avg_UnitPrice           0
Month                   0
Quarter                 0
Year                    0
Is_Peak_Season          0
Target_Demand_Weekly    0
dtype: int64

Test missing values:


Lag_1W                  0
Lag_2W                  0
Lag_4W                  0
Rolling_Mean_4W         0
Rolling_Std_4W          0
Demand_Trend_Ratio      0
Demand_CV               0
Price_Ratio_Vs_Avg      0
Avg_UnitPrice           0
Month                   0
Quarter                 0
Year                    0
Is_Peak_Season          0
Target_Demand_Weekly    0
dtype: int64

In [7]:
# Dùng cùng một tập quan sát cho cả 3 mô hình
train_pd = train_pd.dropna(subset=features + [target]).copy()
test_pd = test_pd.dropna(subset=features + [target]).copy()

X_train = train_pd[features]
y_train = train_pd[target]

X_test = test_pd[features]
y_test = test_pd[target]

print("Final Train:", X_train.shape)
print("Final Test:", X_test.shape)

Final Train: (1580, 13)
Final Test: (400, 13)


## 5. Baseline Model

Use the previous week's demand (Lag_1W) as the forecast for next week's demand.

This naive forecasting method provides a reference point for evaluating
the performance of machine learning models.

In [8]:
baseline_pred = X_test["Lag_1W"].clip(lower=0).to_numpy()

print("Baseline predictions:", len(baseline_pred))
print("Actual observations:", len(y_test))

Baseline predictions: 400
Actual observations: 400


## 6. Model Training

Train two machine learning models:

1. Random Forest Regressor
2. XGBoost Regressor

XGBoost is trained on log-transformed demand to reduce the influence
of highly skewed target values.

The model configurations are kept consistent with the previous experiment.

In [9]:
rf_model = RandomForestRegressor(
    n_estimators=150,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_pred = np.clip(rf_pred, 0, None)

print("Random Forest training completed.")

Random Forest training completed.


In [10]:
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, np.log1p(y_train))

xgb_pred = np.expm1(xgb_model.predict(X_test))
xgb_pred = np.clip(xgb_pred, 0, None)

print("XGBoost training completed.")

XGBoost training completed.


## 7. Model Evaluation

Evaluate model performance using:

- MAE: Mean Absolute Error.
- RMSE: Root Mean Squared Error.
- WAPE: Weighted Absolute Percentage Error.
- Forecast Score: 100% - WAPE.
- Bias: Forecast overestimation or underestimation.

A negative Bias indicates underforecasting, while a positive Bias
indicates overforecasting.

In [11]:
def evaluate_model(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    total_actual = np.sum(y_true)
    wape = (
        np.sum(np.abs(y_true - y_pred)) / total_actual * 100
        if total_actual != 0 else np.nan
    )

    bias = (
        (np.sum(y_pred) - total_actual) / total_actual * 100
        if total_actual != 0 else np.nan
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape,
        "Forecast Score (%)": 100 - wape if not np.isnan(wape) else np.nan,
        "Bias (%)": bias
    }


results = {
    "Baseline": evaluate_model(y_test, baseline_pred),
    "Random Forest": evaluate_model(y_test, rf_pred),
    "XGBoost": evaluate_model(y_test, xgb_pred)
}

evaluation_df = pd.DataFrame(results).T
display(evaluation_df.round(3))

,MAE,RMSE,WAPE (%),Forecast Score (%),Bias (%)
Baseline,392.210,692.459,71.864,28.136,-2.040
Random Forest,361.344,561.125,66.209,33.791,13.303
XGBoost,308.341,606.871,56.497,43.503,-23.768


## 8. Model Comparison

Compare the three models using the same test dataset.

The results are also compared with the previous experiment (V1)
to understand the impact of correcting the SKU selection process.

In [12]:
comparison = evaluation_df.sort_values("WAPE (%)")

display(comparison.round(3))

best_model = comparison.index[0]
print(f"Best model based on WAPE: {best_model}")

,MAE,RMSE,WAPE (%),Forecast Score (%),Bias (%)
XGBoost,308.341,606.871,56.497,43.503,-23.768
Random Forest,361.344,561.125,66.209,33.791,13.303
Baseline,392.210,692.459,71.864,28.136,-2.040


Best model based on WAPE: XGBoost


In [13]:
v1_wape = {
    "Baseline": 77.235,
    "Random Forest": 71.869,
    "XGBoost": 66.262
}

comparison_v1_v2 = pd.DataFrame({
    "V1 WAPE (%)": pd.Series(v1_wape),
    "V2 WAPE (%)": evaluation_df["WAPE (%)"]
})

comparison_v1_v2["WAPE Change (percentage points)"] = (
    comparison_v1_v2["V2 WAPE (%)"] -
    comparison_v1_v2["V1 WAPE (%)"]
)

display(comparison_v1_v2.round(3))

,V1 WAPE (%),V2 WAPE (%),WAPE Change (percentage points)
Baseline,77.235,71.864,-5.371
Random Forest,71.869,66.209,-5.660
XGBoost,66.262,56.497,-9.765


## 9. Error Analysis

Analyze prediction errors at SKU level to identify products
with high forecasting errors.

Investigate underforecasting, overforecasting, and demand patterns
to understand model limitations.

In [14]:
predictions = test_df.select([
    "Week_Start",
    "StockCode",
    "Weekly_Demand",
    target
]).to_pandas()

predictions["Baseline_Pred"] = baseline_pred
predictions["RF_Pred"] = rf_pred
predictions["XGB_Pred"] = xgb_pred

predictions["XGB_Error"] = (
    predictions[target] - predictions["XGB_Pred"]
)

predictions["XGB_Abs_Error"] = predictions["XGB_Error"].abs()

predictions["XGB_Error_Type"] = np.where(
    predictions["XGB_Error"] > 0,
    "Underforecast",
    np.where(predictions["XGB_Error"] < 0, "Overforecast", "Accurate")
)

display(predictions.head())

,Week_Start,StockCode,Weekly_Demand,Target_Demand_Weekly,Baseline_Pred,RF_Pred,XGB_Pred,XGB_Error,XGB_Abs_Error,XGB_Error_Type
0,2011-07-18,15036,510,638,352,533.546667,309.648102,328.351898,328.351898,Underforecast
1,2011-07-25,15036,638,943,510,740.520000,344.693451,598.306549,598.306549,Underforecast
2,2011-08-01,15036,943,268,638,708.046667,507.915833,-239.915833,239.915833,Overforecast
3,2011-08-08,15036,268,60,943,614.013333,535.714722,-475.714722,475.714722,Overforecast
4,2011-08-15,15036,60,60,268,870.053333,519.841858,-459.841858,459.841858,Overforecast


In [15]:
sku_errors = (
    predictions
    .groupby("StockCode")
    .apply(lambda g: pd.Series({
        "Actual_Total": g[target].sum(),
        "Predicted_Total": g["XGB_Pred"].sum(),
        "MAE": g["XGB_Abs_Error"].mean(),
        "WAPE (%)": (
            g["XGB_Abs_Error"].sum() / g[target].sum() * 100
            if g[target].sum() != 0 else np.nan
        ),
        "Bias (%)": (
            (g["XGB_Pred"].sum() - g[target].sum())
            / g[target].sum() * 100
            if g[target].sum() != 0 else np.nan
        )
    }))
    .reset_index()
    .sort_values("WAPE (%)", ascending=False)
)

display(sku_errors)

,StockCode,Actual_Total,Predicted_Total,MAE,WAPE (%),Bias (%)
14,84568,7676.0,2351.521973,402.851471,104.963906,-69.365269
0,15036,4480.0,5174.813477,210.925854,94.163328,15.509229
12,23166,2251.0,3031.585693,95.041256,84.443586,34.677285
7,21982,2242.0,2736.457031,88.519991,78.965202,22.054283
1,17003,7566.0,5639.578125,294.121400,77.748189,-25.461563
4,21213,6415.0,7121.574219,212.495972,66.249719,11.014407
9,22197,34637.0,14680.909180,1062.657620,61.359680,-57.614952
17,84991,7334.0,7255.824219,220.836450,60.222648,-1.065936
11,22616,9374.0,6125.923828,281.125018,59.979735,-34.649842
6,21977,8367.0,8408.086914,246.545356,58.932797,0.491059


In [16]:
error_summary = (
    predictions["XGB_Error_Type"]
    .value_counts()
    .rename_axis("Error_Type")
    .reset_index(name="Count")
)

display(error_summary)

,Error_Type,Count
0,Overforecast,219
1,Underforecast,181


In [17]:
# Ghép thông tin lịch sử và kết quả dự báo theo đúng thứ tự Test
error_detail = test_df.select([
    "Week_Start",
    "StockCode",
    "Weekly_Demand",
    "Lag_1W",
    "Target_Demand_Weekly"
]).to_pandas()

error_detail["XGB_Pred"] = xgb_pred

# Sai số dự báo
error_detail["Error"] = (
    error_detail["Target_Demand_Weekly"] - error_detail["XGB_Pred"]
)

# Chỉ xem 3 SKU cần điều tra
focus_skus = ["84568", "15036", "22197"]

for sku in focus_skus:
    print(f"\n========== SKU {sku} ==========")

    display(
        error_detail[
            error_detail["StockCode"] == sku
        ][[
            "Week_Start",
            "Weekly_Demand",
            "Lag_1W",
            "Target_Demand_Weekly",
            "XGB_Pred",
            "Error"
        ]].round(2)
    )


========== SKU 84568 ==========


/tmp/ipykernel_100165/4112701046.py:33: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].round(2)


,Week_Start,Weekly_Demand,Lag_1W,Target_Demand_Weekly,XGB_Pred,Error
280,2011-07-18,44,583,4384,109.800003,4274.20
281,2011-07-25,4384,44,340,158.210007,181.79
282,2011-08-01,340,4384,35,202.110001,-167.11
283,2011-08-08,35,340,0,481.029999,-481.03
284,2011-08-15,0,35,0,206.880005,-206.88
285,2011-08-22,0,0,1532,22.680000,1509.32
286,2011-08-29,1532,0,45,25.969999,19.03
287,2011-09-05,45,1532,32,141.580002,-109.58
288,2011-09-12,32,45,24,99.459999,-75.46
289,2011-09-19,24,32,9,114.330002,-105.33



========== SKU 15036 ==========


/tmp/ipykernel_100165/4112701046.py:33: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].round(2)


,Week_Start,Weekly_Demand,Lag_1W,Target_Demand_Weekly,XGB_Pred,Error
0,2011-07-18,510,352,638,309.649994,328.35
1,2011-07-25,638,510,943,344.690002,598.31
2,2011-08-01,943,638,268,507.920013,-239.92
3,2011-08-08,268,943,60,535.710022,-475.71
4,2011-08-15,60,268,60,519.840027,-459.84
5,2011-08-22,60,60,181,295.559998,-114.56
6,2011-08-29,181,60,215,120.849998,94.15
7,2011-09-05,215,181,64,161.470001,-97.47
8,2011-09-12,64,215,93,215.300003,-122.30
9,2011-09-19,93,64,30,182.279999,-152.28



========== SKU 22197 ==========


/tmp/ipykernel_100165/4112701046.py:33: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].round(2)


,Week_Start,Weekly_Demand,Lag_1W,Target_Demand_Weekly,XGB_Pred,Error
180,2011-07-18,621,157,436,375.940002,60.06
181,2011-07-25,436,621,2178,296.040009,1881.96
182,2011-08-01,2178,436,1735,451.850006,1283.15
183,2011-08-08,1735,2178,518,669.719971,-151.72
184,2011-08-15,518,1735,826,700.000000,126.00
185,2011-08-22,826,518,956,717.369995,238.63
186,2011-08-29,956,826,516,717.469971,-201.47
187,2011-09-05,516,956,570,865.349976,-295.35
188,2011-09-12,570,516,682,664.109985,17.89
189,2011-09-19,682,570,1675,625.739990,1049.26


In [18]:
sku = "84568"

sku_history = (
    df_gold
    .filter(pl.col("StockCode") == sku)
    .sort("Week_Start")
    .select(["Week_Start", "Weekly_Demand", "Target_Demand_Weekly"])
    .to_pandas()
)

print("Số tuần:", len(sku_history))
print("Số tuần có nhu cầu:", (sku_history["Weekly_Demand"] > 0).sum())
print("Tỷ lệ tuần không có nhu cầu:", (sku_history["Weekly_Demand"] == 0).mean() * 100)
print("\nThống kê nhu cầu:")
print(sku_history["Weekly_Demand"].describe())

print("\nTop 10 tuần có nhu cầu cao nhất:")
display(
    sku_history.nlargest(10, "Weekly_Demand")
    .round(2)
)

Số tuần: 99
Số tuần có nhu cầu: 87
Tỷ lệ tuần không có nhu cầu: 12.121212121212121

Thống kê nhu cầu:
count      99.000000
mean      374.676768
std       681.729730
min         0.000000
25%        19.000000
50%        57.000000
75%       338.500000
max      4384.000000
Name: Weekly_Demand, dtype: float64

Top 10 tuần có nhu cầu cao nhất:


/tmp/ipykernel_100165/2513330551.py:20: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,Week_Start,Weekly_Demand,Target_Demand_Weekly
80,2011-07-25,4384,340
41,2010-10-18,2899,293
38,2010-09-27,2044,322
10,2010-03-15,2042,1194
9,2010-03-08,1780,2042
31,2010-08-09,1767,41
63,2011-03-28,1740,30
85,2011-08-29,1532,45
46,2010-11-22,1452,0
40,2010-10-11,1443,2899


In [19]:
sku_history["Month"] = pd.to_datetime(sku_history["Week_Start"]).dt.month

monthly_stats = (
    sku_history.groupby("Month")["Weekly_Demand"]
    .agg(["count", "mean", "median", "std", "max"])
    .round(2)
)

display(monthly_stats)

,count,mean,median,std,max
Month,,,,,
1,9,24.56,19.0,29.31,96
2,8,265.62,297.5,237.93,581
3,9,895.44,596.0,798.95,2042
4,8,199.38,33.0,270.44,620
5,10,158.00,153.5,160.35,362
6,8,300.12,288.0,264.05,872
7,8,667.75,35.5,1515.59,4384
8,10,385.00,42.5,675.97,1767
9,8,398.75,40.5,701.55,2044


In [20]:
spikes = sku_history[
    sku_history["Weekly_Demand"] >= 1000
].copy()

spikes["Week_Start"] = pd.to_datetime(spikes["Week_Start"])
spikes["Gap_Weeks"] = (
    spikes["Week_Start"].diff().dt.days / 7
)

display(spikes)

,Week_Start,Weekly_Demand,Target_Demand_Weekly,Month,Gap_Weeks
9,2010-03-08,1780,2042,3,NaN
10,2010-03-15,2042,1194,3,1.0
11,2010-03-22,1194,596,3,1.0
31,2010-08-09,1767,41,8,20.0
38,2010-09-27,2044,322,9,7.0
40,2010-10-11,1443,2899,10,2.0
41,2010-10-18,2899,293,10,1.0
46,2010-11-22,1452,0,11,5.0
63,2011-03-28,1740,30,3,18.0
80,2011-07-25,4384,340,7,17.0


In [21]:
sku = "84568"

comparison = predictions[
    predictions["StockCode"] == sku
].copy()

comparison["Baseline_Error"] = (
    comparison["Target_Demand_Weekly"] - comparison["Baseline_Pred"]
)

comparison["XGB_Error"] = (
    comparison["Target_Demand_Weekly"] - comparison["XGB_Pred"]
)

display(
    comparison[[
        "Week_Start",
        "Target_Demand_Weekly",
        "Baseline_Pred",
        "XGB_Pred",
        "Baseline_Error",
        "XGB_Error"
    ]].round(2)
)

/tmp/ipykernel_100165/624077969.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].round(2)


,Week_Start,Target_Demand_Weekly,Baseline_Pred,XGB_Pred,Baseline_Error,XGB_Error
280,2011-07-18,4384,583,109.800003,3801,4274.20
281,2011-07-25,340,44,158.210007,296,181.79
282,2011-08-01,35,4384,202.110001,-4349,-167.11
283,2011-08-08,0,340,481.029999,-340,-481.03
284,2011-08-15,0,35,206.880005,-35,-206.88
285,2011-08-22,1532,0,22.680000,1532,1509.32
286,2011-08-29,45,0,25.969999,45,19.03
287,2011-09-05,32,1532,141.580002,-1500,-109.58
288,2011-09-12,24,45,99.459999,-21,-75.46
289,2011-09-19,9,32,114.330002,-23,-105.33


In [24]:
# 1. Sắp xếp dữ liệu theo SKU và thời gian
weekly_check = (
    df_gold
    .sort(["StockCode", "Week_Start"])
    .with_columns(
        pl.col("Week_Start")
        .diff()
        .over("StockCode")
        .alias("Week_Gap")
    )
)

# 2. Kiểm tra tổng quan
print("Tổng số dòng:", weekly_check.height)
print("Số SKU:", weekly_check["StockCode"].n_unique())

# 3. Kiểm tra khoảng cách tuần bất thường
gap_errors = weekly_check.filter(
    pl.col("Week_Gap").is_not_null() &
    (pl.col("Week_Gap").dt.total_days() != 7)
)

print("Số khoảng cách tuần bất thường:", gap_errors.height)
display(gap_errors.select([
    "StockCode", "Week_Start", "Week_Gap"
]).head(20))

# 4. Kiểm tra SKU-tuần bị trùng
duplicate_weeks = (
    df_gold.group_by(["StockCode", "Week_Start"])
    .len()
    .filter(pl.col("len") > 1)
)

print("Số SKU-tuần bị trùng:", duplicate_weeks.height)
display(duplicate_weeks.head(20))

Tổng số dòng: 486684
Số SKU: 4916
Số khoảng cách tuần bất thường: 4916


StockCode,Week_Start,Week_Gap
str,datetime[μs],duration[μs]
"""10002""",2011-01-03 00:00:00,14d
"""10002R""",2011-01-03 00:00:00,14d
"""10080""",2011-01-03 00:00:00,14d
"""10109""",2011-01-03 00:00:00,14d
"""10120""",2011-01-03 00:00:00,14d
…,…,…
"""15030""",2011-01-03 00:00:00,14d
"""15034""",2011-01-03 00:00:00,14d
"""15036""",2011-01-03 00:00:00,14d


Số SKU-tuần bị trùng: 0


StockCode,Week_Start,len
str,datetime[μs],u32


In [23]:
# Liệt kê các biến DataFrame Polars đang tồn tại
for name, value in list(globals().items()):
    if isinstance(value, pl.DataFrame):
        print(f"{name}: {value.shape}")

df_gold: (486684, 19)
train_history: (388364, 19)
sku_ranking: (4445, 3)
top_skus: (20, 1)
df_sub: (1980, 19)
train_df: (1580, 19)
test_df: (400, 19)


In [25]:
# 1. Xác định ngày trước đó và khoảng cách giữa hai tuần
gap_summary = (
    df_gold.sort(["StockCode", "Week_Start"])
    .with_columns([
        pl.col("Week_Start").shift(1).over("StockCode").alias("Previous_Week"),
        pl.col("Week_Start").diff().over("StockCode").dt.total_days().alias("Gap_Days")
    ])
    .filter(pl.col("Gap_Days").is_not_null() & (pl.col("Gap_Days") != 7))
    .group_by(["Previous_Week", "Week_Start", "Gap_Days"])
    .agg([
        pl.col("StockCode").n_unique().alias("Affected_SKUs"),
        pl.len().alias("Affected_Rows")
    ])
    .sort("Previous_Week")
)

display(gap_summary)

Previous_Week,Week_Start,Gap_Days,Affected_SKUs,Affected_Rows
datetime[μs],datetime[μs],i64,u32,u32
2010-12-20 00:00:00,2011-01-03 00:00:00,14,4916,4916


In [26]:

calendar_check = (
    df_gold.select("Week_Start")
    .unique()
    .sort("Week_Start")
    .with_columns(
        pl.col("Week_Start").diff().dt.total_days().alias("Gap_Days")
    )
)

display(
    calendar_check.filter(
        pl.col("Gap_Days").is_not_null() &
        (pl.col("Gap_Days") != 7)
    )
)

Week_Start,Gap_Days
datetime[μs],i64
2011-01-03 00:00:00,14


In [27]:
print(df_gold.schema)

display(
    df_gold.filter(
        (pl.col("StockCode") == "10002") &
        pl.col("Week_Start").is_between(
            pl.datetime(2010, 12, 14),
            pl.datetime(2011, 1, 10)
        )
    ).sort("Week_Start")
)

Schema({'Week_Start': Datetime(time_unit='us', time_zone=None), 'StockCode': String, 'Weekly_Demand': Int64, 'Weekly_Revenue': Float64, 'Avg_UnitPrice': Float64, 'Description': String, 'Target_Demand_Weekly': Int64, 'Lag_1W': Int64, 'Lag_2W': Int64, 'Lag_4W': Int64, 'Rolling_Mean_4W': Float64, 'Rolling_Std_4W': Float64, 'Month': Int8, 'Quarter': Int8, 'Year': Int32, 'Is_Peak_Season': Int32, 'Demand_Trend_Ratio': Float64, 'Demand_CV': Float64, 'Price_Ratio_Vs_Avg': Float64})


Week_Start,StockCode,Weekly_Demand,Weekly_Revenue,Avg_UnitPrice,Description,Target_Demand_Weekly,Lag_1W,Lag_2W,Lag_4W,Rolling_Mean_4W,Rolling_Std_4W,Month,Quarter,Year,Is_Peak_Season,Demand_Trend_Ratio,Demand_CV,Price_Ratio_Vs_Avg
datetime[μs],str,i64,f64,f64,str,i64,i64,i64,i64,f64,f64,i8,i8,i32,i32,f64,f64,f64
2010-12-20 00:00:00,"""10002""",2,3.32,1.66,"""INFLATABLE POLITICAL GLOBE""",73,41,228,51,120.25,90.086532,12,4,2010,1,0.338144,0.742982,1.439574
2011-01-03 00:00:00,"""10002""",73,62.86,1.12,"""INFLATABLE POLITICAL GLOBE""",42,2,41,161,108.0,104.775951,1,1,2011,0,0.018349,0.961247,0.864953
2011-01-10 00:00:00,"""10002""",42,35.7,0.85,"""INFLATABLE POLITICAL GLOBE""",76,73,2,228,86.0,99.018517,1,1,2011,0,0.83908,1.138144,0.646525


In [29]:
print("df_silver tồn tại:", "df_silver" in globals())

df_silver tồn tại: False


In [30]:
from pathlib import Path

silver_path = Path("../data/2_silver")

print("Thư mục Silver tồn tại:", silver_path.exists())

if silver_path.exists():
    for file in silver_path.rglob("*"):
        if file.is_file():
            print(file)

Thư mục Silver tồn tại: True
../data/2_silver/sheet_2010_2011_cleaned.parquet
../data/2_silver/sheet_2009_2010_cleaned.parquet


In [32]:
df_silver = pl.read_parquet("../data/2_silver/sheet_2010_2011_cleaned.parquet")

print("Shape:", df_silver.shape)
print("Schema:", df_silver.schema)
display(df_silver.head(5))

Shape: (530105, 8)
Schema: Schema({'Invoice': String, 'StockCode': String, 'Description': String, 'Quantity': Int64, 'InvoiceDate': Datetime(time_unit='us', time_zone=None), 'UnitPrice': Float64, 'CustomerID': String, 'Country': String})


Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,datetime[μs],f64,str,str
"""536365""","""85123A""","""WHITE HANGING HEART T-LIGHT HO…",6,2010-12-01 08:26:00,2.55,"""17850""","""United Kingdom"""
"""536365""","""71053""","""WHITE METAL LANTERN""",6,2010-12-01 08:26:00,3.39,"""17850""","""United Kingdom"""
"""536365""","""84406B""","""CREAM CUPID HEARTS COAT HANGER""",8,2010-12-01 08:26:00,2.75,"""17850""","""United Kingdom"""
"""536365""","""84029G""","""KNITTED UNION FLAG HOT WATER B…",6,2010-12-01 08:26:00,3.39,"""17850""","""United Kingdom"""
"""536365""","""84029E""","""RED WOOLLY HOTTIE WHITE HEART.""",6,2010-12-01 08:26:00,3.39,"""17850""","""United Kingdom"""


In [33]:
print(df_silver.schema)
display(df_silver.head(5))

Schema({'Invoice': String, 'StockCode': String, 'Description': String, 'Quantity': Int64, 'InvoiceDate': Datetime(time_unit='us', time_zone=None), 'UnitPrice': Float64, 'CustomerID': String, 'Country': String})


Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,datetime[μs],f64,str,str
"""536365""","""85123A""","""WHITE HANGING HEART T-LIGHT HO…",6,2010-12-01 08:26:00,2.55,"""17850""","""United Kingdom"""
"""536365""","""71053""","""WHITE METAL LANTERN""",6,2010-12-01 08:26:00,3.39,"""17850""","""United Kingdom"""
"""536365""","""84406B""","""CREAM CUPID HEARTS COAT HANGER""",8,2010-12-01 08:26:00,2.75,"""17850""","""United Kingdom"""
"""536365""","""84029G""","""KNITTED UNION FLAG HOT WATER B…",6,2010-12-01 08:26:00,3.39,"""17850""","""United Kingdom"""
"""536365""","""84029E""","""RED WOOLLY HOTTIE WHITE HEART.""",6,2010-12-01 08:26:00,3.39,"""17850""","""United Kingdom"""


In [34]:
missing_week = df_silver.filter(
    (pl.col("InvoiceDate") >= pl.datetime(2010, 12, 27)) &
    (pl.col("InvoiceDate") < pl.datetime(2011, 1, 3))
)

print("Số giao dịch:", missing_week.height)
print("Số SKU:", missing_week["StockCode"].n_unique())

Số giao dịch: 0
Số SKU: 0
